# CellChat pathology overview across six regions

Regenerates `02all_region_compact_overview.pdf` from the finalized pair-summary table. The plotted table excludes Vasc as both sender and receiver. Each tile color is the median donor-level pathology slope; the overlaid count is the number of ligand–receptor features with pair-specific BH FDR < 0.05.

Adjust the font-size and spacing settings in the first code cell, then run all cells to regenerate the table, PDF, and PNG.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Find the project root whether launched from the repository or this folder.
HERE = Path.cwd().resolve()
PROJECT_ROOT = (Path(os.environ['REGA_SHARED_ROOT']) / 'part8_cellchat') if os.environ.get('REGA_SHARED_ROOT') else next(p for p in (HERE, *HERE.parents) if (p / 'finalized/tables/pathology_progression_tests.tsv').exists())

FIGURE_DIR = str(PROJECT_ROOT / 'finalized/figures')
TABLE_DIR = str(PROJECT_ROOT / 'finalized/tables')
INPUT_TSV =str(PROJECT_ROOT / 'finalized/tables/all_pair_overview_summary.tsv')
PLOT_TABLE =str(PROJECT_ROOT / 'finalized/tables/all_pair_overview_summary.tsv')
PDF_PATH = str(PROJECT_ROOT / 'finalized/figures/02all_region_compact_overview.pdf')
PNG_PATH = str(PROJECT_ROOT / 'finalized/figures/02all_region_compact_overview.png')

REGIONS = ['AG', 'EC', 'HC', 'MTC', 'PFC', 'TH']
CELLTYPES = ['Exc', 'Inh', 'Ast', 'Oli', 'OPC', 'Mic_Immune']
CELLTYPE_LABELS = ['Mic' if celltype == 'Mic_Immune' else celltype for celltype in CELLTYPES]

# Figure controls: increase font sizes or reduce gaps here.
FIGURE_SIZE = (9, 4.5)
TITLE_FONTSIZE = 20
REGION_FONTSIZE = 16
TICK_FONTSIZE = 10
AXIS_LABEL_FONTSIZE = 12
ANNOTATION_FONTSIZE = 9
COLORBAR_FONTSIZE = 11
WSPACE = 0.08
HSPACE = 0.16
DPI = 300
print('Project root:', PROJECT_ROOT)
print('PDF output:', PDF_PATH)

In [ ]:
overview = pd.read_csv(INPUT_TSV, sep='\t')
overview = overview[
    overview['region'].isin(REGIONS)
    & overview['source'].isin(CELLTYPES)
    & overview['target'].isin(CELLTYPES)
].copy()
overview['region'] = pd.Categorical(overview['region'], REGIONS, ordered=True)
overview['source'] = pd.Categorical(overview['source'], CELLTYPES, ordered=True)
overview['target'] = pd.Categorical(overview['target'], CELLTYPES, ordered=True)
overview = overview.sort_values(['region', 'source', 'target']).reset_index(drop=True)
expected = len(REGIONS) * len(CELLTYPES) ** 2
if len(overview) != expected:
    raise ValueError(f'Expected {expected} region × sender × receiver rows after filtering; found {len(overview)}')
if overview[['median_beta', 'n_q_lt_0_05']].isna().any().any():
    raise ValueError('Missing median slopes or significant-feature counts in plotting table')
overview.to_csv(PLOT_TABLE, sep='\t', index=False)
print(f'Wrote {len(overview)} rows to {PLOT_TABLE}')
print('Rows per region:', overview.groupby('region', observed=False).size().to_dict())
display(overview.head())

In [ ]:
limit = float(np.nanmax(np.abs(overview['median_beta'].to_numpy())))
if not np.isfinite(limit) or limit == 0:
    limit = 1e-12
norm = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)
fig, axes = plt.subplots(2, 3, figsize=FIGURE_SIZE, sharex=True, sharey=True)
fig.subplots_adjust(left=0.075, right=0.91, bottom=0.22, top=0.78, wspace=WSPACE, hspace=0.38)
cmap = plt.get_cmap('coolwarm')
for ax, region in zip(axes.flat, REGIONS):
    d = overview[overview['region'] == region]
    matrix = d.pivot(index='source', columns='target', values='median_beta').reindex(index=CELLTYPES, columns=CELLTYPES)
    counts = d.pivot(index='source', columns='target', values='n_q_lt_0_05').reindex(index=CELLTYPES, columns=CELLTYPES)
    im = ax.imshow(matrix.to_numpy(dtype=float), cmap=cmap, norm=norm, aspect='equal', interpolation='nearest')
    ax.set_title(region, fontsize=REGION_FONTSIZE, fontweight='bold', pad=6)
    ax.set_xticks(range(len(CELLTYPES)), CELLTYPE_LABELS, rotation=42, ha='right', fontsize=TICK_FONTSIZE)
    ax.set_yticks(range(len(CELLTYPES)), CELLTYPE_LABELS, fontsize=TICK_FONTSIZE)
    ax.set_xticks(np.arange(-.5, len(CELLTYPES), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(CELLTYPES), 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.9)
    ax.tick_params(which='minor', bottom=False, left=False)
    for i in range(len(CELLTYPES)):
        for j in range(len(CELLTYPES)):
            count = int(counts.iloc[i, j])
            if count > 0:
                ax.text(j, i, str(count), color='black', ha='center', va='center', fontsize=ANNOTATION_FONTSIZE, fontweight='bold', zorder=4)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
fig.suptitle('CellChat pathology overview across six regions', fontsize=TITLE_FONTSIZE, fontweight='bold', y=0.985)
fig.text(0.5, 0.90, 'Tile color: median pathology slope; black number: pair-FDR-significant feature count.', ha='center', va='center', fontsize=AXIS_LABEL_FONTSIZE)
fig.text(0.015, 0.49, 'Sender', rotation=90, va='center', ha='center', fontsize=AXIS_LABEL_FONTSIZE)
fig.text(0.49, 0.012, 'Receiver', va='center', ha='center', fontsize=AXIS_LABEL_FONTSIZE)
cax = fig.add_axes([0.935, 0.18, 0.018, 0.64])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label('Median pathology slope\nper ligand–receptor feature', fontsize=COLORBAR_FONTSIZE)
cbar.ax.tick_params(labelsize=TICK_FONTSIZE)
fig.savefig(PDF_PATH, bbox_inches='tight')
fig.savefig(PNG_PATH, dpi=DPI, bbox_inches='tight')
plt.show()
print('Saved:', PDF_PATH, 'and', PNG_PATH)